# Part 1: Build knowledge bases with agentic retrieval

Cocoarynth is a chocolate retailer preparing an Origin Passport export for Mapayca Markets. As a developer, you need to combine partner requirements, traceability policy, and engineering standards without treating any of them as proof of what the application currently implements.

**Mission**
- Upload Cocoarynth's wholesale traceability data-sharing policy to a **file knowledge source** and query its controls.
- Connect the **product and traceability documents** and **engineering practices** indexes as **indexed knowledge sources**.
- Build a two-source **knowledge base** that grounds implementation guidance in the right documents.

## The building blocks

In Foundry IQ (Azure AI Search), a **knowledge base** connects a chat completion model to one or more knowledge sources. It defines which sources to query, which model to use for reasoning, and how retrieval should work.

In this notebook, you'll explore two types of knowledge sources:

1. **File knowledge source**: Upload raw files directly. The service handles chunking, embedding, and indexing automatically.
2. **Indexed knowledge source**: Connect to a pre-built search index with control over schema and processing.

The policy file is also included in the product-document index. The standalone policy KB demonstrates file ingestion; the combined KB uses only the two indexed sources.

## Pre-configured environment

This lab uses the following Azure resources:

* Foundry IQ (Azure AI Search), serverless in West Central US
  * `cocoarynth-documents-index`: four PDFs covering pilot requirements, immutable Origin Passport snapshots, data-sharing policy, and support procedures.
  * `cocoarynth-engineering-practices-index`: four PDFs covering API/export contracts, React/TypeScript, accessibility, and engineering culture.
  * Semantic ranking and 3072-dimensional embeddings.
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-5.4`: Reasoning and answer synthesis.
  * `text-embedding-3-large`: Vector embeddings.

Both indexes must be populated before you start. Maintainers ingest the PDFs and export snapshots once; subsequent provisioning restores those snapshots. Chunk counts depend on ingestion and are not fixed PDF counts. See [the setup guide](../../deploy_yourself.md#prepare-the-cocoarynth-indexes).

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Foundry IQ (Azure AI Search) endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os
from pathlib import Path

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

REPO_ROOT = next(
    (
        directory
        for directory in (Path.cwd(), *Path.cwd().parents)
        if (directory / ".env.sample").is_file() and (directory / "infra").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")
if not (REPO_ROOT / ".env").is_file():
    raise RuntimeError("Provision the lab environment before running this notebook.")
load_dotenv(REPO_ROOT / ".env", override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

DOCUMENTS_INDEX = "cocoarynth-documents-index"
ENGINEERING_PRACTICES_INDEX = "cocoarynth-engineering-practices-index"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")

---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly. The service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "cocoarynth-policy-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Cocoarynth wholesale traceability data-sharing policy defining permitted fields, prohibited information, separate approvals, and sharing controls",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Upload the wholesale traceability data-sharing policy directly to the file knowledge source. The service extracts text, chunks the document, and generates embeddings.

Ingestion time varies. If the first retrieval reports that the file is not ready, allow ingestion to finish and rerun the retrieval cell.


In [ ]:
data_dir = REPO_ROOT / "data"
files_to_upload = [
    data_dir / "cocoarynth" / "documents" / "wholesale-traceability-data-sharing-policy.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as pdf_file:
        file_content = pdf_file.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Create a knowledge base for the file source

A knowledge base is the orchestration layer that combines:

1. Your data sources (the file knowledge source)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM will generate a complete natural language answer from the retrieved chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "cocoarynth-policy-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    description="Cocoarynth wholesale traceability policy knowledge base",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: Query the file knowledge base

Ask a question about the uploaded document. The knowledge base will search across the file content and synthesize a natural language answer with citations.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "Which Origin Passport fields may Cocoarynth share with wholesale partners, which are prohibited, and which need separate approval?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

The activity log shows the sequence of steps taken by the knowledge base. Each activity contains:

* `type`: The type of activity. For this log, you will see the first step of "modelQueryPlanning", multiple "file" steps (one for each query sent to the file knowledge source), "modelAnswerSynthesis" for the answer synthesis step, and "agenticReasoning" for the semantic re-ranking step.
* `elapsedMS`: The duration of the activity.
* `id`: The activity ID, which is helpful to connect each reference to the activity that produced it. You'll review references next.

The activity entries also contain other metadata specific to their type, so that you can see how many tokens were used by any model-powered steps and see the queries sent to sources for retrieval steps.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references show which sources were retrieved, and the source metadata for each reference. Every result contains:

* `type`: Always "file", since this knowledge base only contains a file source.
* `id`: A numeric identifier for the reference. This is what appears in the answer as citation markers like `[ref:1]`.
* `activitySource`: The id of the retrieval activity in the activity log that produced this reference. You can use this to trace which search step returned this document.
* `sourceData`: Fields from the retrieved chunk in the auto-created search index - always `uid`, `snippet`, and `metadata_storage_path` for file source indexes.
* `rerankerScore`: A score between 0-4 from the re-ranking model, where 4 is the best.
* `docName`: The auto-generated filename for the uploaded file.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed knowledge sources

**Indexed knowledge sources** connect to pre-built search indexes, giving you control over schema, chunking, and embeddings.

`cocoarynth-documents-index` contains Cocoarynth's pilot requirements, Origin Passport architecture decision, data-sharing policy, and support runbook. `cocoarynth-engineering-practices-index` contains API/export contracts, React/TypeScript guidance, accessibility guidance, and engineering culture material.

## Step 6: Verify search indexes

Confirm that both indexes exist and contain chunks. Each corpus has four PDFs; Content Understanding produces multiple chunks per PDF, so the final counts depend on ingestion. An empty index is not ready for the exercises.

In [ ]:
from azure.search.documents import SearchClient

for index_name in [DOCUMENTS_INDEX, ENGINEERING_PRACTICES_INDEX]:
    client = SearchClient(
        endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
        index_name=index_name,
        credential=search_credential,
    )
    count = client.get_document_count()
    if count == 0:
        client.close()
        raise RuntimeError(f"Index '{index_name}' is empty. Complete document ingestion or snapshot restoration first.")
    print(f"{index_name}: {count} document chunks")
    client.close()

## Step 7: Create two indexed knowledge sources

Create one knowledge source for each populated index:

* `cocoarynth-documents-source`: Pilot requirements, Origin Passport architecture, data-sharing policy, and the support runbook.
* `cocoarynth-engineering-practices-source`: API/export contracts, React/TypeScript, accessibility, and engineering culture guidance.

Both sources search the `chunk` field and return `blob_path`, `title`, `page_number_from`, and `page_number_to` as citation metadata. Image extraction is not part of this lab's portable index schema.


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "cocoarynth-documents-source"
ENGINEERING_PRACTICES_KNOWLEDGE_SOURCE_NAME = "cocoarynth-engineering-practices-source"
KNOWLEDGE_SOURCE_NAMES = [
    DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    ENGINEERING_PRACTICES_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (
        DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        DOCUMENTS_INDEX,
        "Cocoarynth product requirements, immutable Origin Passport architecture, data-sharing policy, and support runbook",
    ),
    (
        ENGINEERING_PRACTICES_KNOWLEDGE_SOURCE_NAME,
        ENGINEERING_PRACTICES_INDEX,
        "Cocoarynth engineering, API, frontend, accessibility, and engineering culture guidance",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")

## Step 8: Create the multi-source knowledge base

Create `cocoarynth-kb-all` with the product-document source and the engineering-practice source, matching the chocolate-shop lab's two-source setup. The standalone file KB remains a separate exercise; the data-sharing policy is already present in the product-document index.

This notebook uses `ANSWER_SYNTHESIS` so the model returns a grounded natural-language answer. The original developer KB app uses extractive output for its client to consume.


In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "cocoarynth-kb-all"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Cocoarynth knowledge base for product requirements, traceability policy, and engineering practices",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: Query the multi-source knowledge base

Ask a question that needs both the partner requirements and the engineering standards:

* _"Mapayca Markets requires CSV Origin Passport exports. What fields and format must the export use, and which API contract and test practices should guide its implementation?"_

Then try:

* _"Can Cocoarynth send JSON renamed to .csv for the Mapayca pilot, and does the pilot require bulk export?"_
* _"How should CSV export preserve immutable Origin Passport snapshots and exclude confidential fields?"_
* _"What accessibility and React/TypeScript practices should guide an Origin Passport export control?"_

Use the activity log and references to distinguish partner requirements, sharing policy, architecture decisions, and engineering guidance. No application repository is indexed here: these documents do not prove what the current application implements.


In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
)

documents_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
engineering_practices_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=ENGINEERING_PRACTICES_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "Mapayca Markets requires CSV Origin Passport exports. "
    "What fields and format must the export use, and which API contract and test practices should guide its implementation?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[documents_params, engineering_practices_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

Look for `searchIndex` activities for the product-document and engineering-practice sources. A cross-source question may retrieve both partner requirements and company engineering standards. The combined KB does not include the standalone file source.


In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

References identify which knowledge source supports each claim. The combined KB's indexed references have `type: "searchIndex"` and include `title`, `blob_path`, and page numbers. Trace requirements to the partner document and implementation guidance to the relevant engineering standard.


In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### Source hunt

Review the references and activity log above:

1. Which document requires CSV rather than JSON and defines the eleven export columns?
2. Which policy excludes commercial terms, personal contact information, internal quality notes, and employee comments?
3. Which engineering guide defines download headers, CSV escaping, and contract tests?
4. Do the references expose `title`, `blob_path`, `page_number_from`, and `page_number_to`?
5. Did the answer distinguish requirements and engineering guidance from claims about the current implementation?


## Bonus: Query from the GitHub Copilot App

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or the GitHub Copilot App. Run the cell below to print the MCP URL for the Cocoarynth knowledge base, then follow the steps after it to query the knowledge base from the GitHub Copilot App.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(mcp_url)

### Sign in to GitHub

Sign in through the [Skillable Events GitHub enterprise](https://github.com/enterprises/skillable-events/sso).

You will be signed into a special account created for the lab environment, not your actual GitHub account.

### Start the GitHub Copilot App

1. Open the GitHub Copilot App by finding the shortcut on the desktop.
2. Select "Sign in to GitHub" button.
3. When the browser pops up, paste the device code that was automatically copied into your clipboard, and select "Authorize github".
4. Return to the app. You don't need to connect any repositories, so select "Continue" on repository selection screen and "Finish" on the preferences screen.


### Add the knowledge base MCP server

1. Select **Customize** in the sidebar.
2. Select the **MCP** tab.
3. Open the **Add** menu, then select **MCP server**.
4. Fill in the details for the Cocoarynth knowledge base:
   * Server name: `cocoarynth-kb`
   * Server type: **HTTP** (*not* Local/SSE)
   * URL: Paste the URL printed by the cell above.
   * Headers: Enter `api-key` for the header name and the value of `AZURE_SEARCH_ADMIN_KEY` from your `.env` file for its value.
5. Leave other fields at their defaults, then select **Add server**.
6. Confirm that `cocoarynth-kb` appears in the list of MCP servers.

### Ask a question grounded by the knowledge base

Create a new chat by selecting **+** next to **Chats** in the sidebar, then ask a question that matches this notebook. For example:

_"What fields, CSV format, and API contract tests are required for the Mapayca Markets Origin Passport pilot?"_


## Mission complete

**What you built:**

* **File knowledge source**: Cocoarynth's wholesale traceability data-sharing policy uploaded directly for permitted-field and approval questions.
* **Indexed knowledge sources**: Product/traceability documents and engineering practices with portable citation metadata.
* **Multi-source knowledge base**: `cocoarynth-kb-all`, combining the two indexed sources to ground implementation guidance in requirements, policy, and engineering standards.
